# Swissquote BBO — spreads EDA

Mid price, absolute spread (`spread_abs`), and spread in basis points (`spread_bps`), plus a deep dive into Swissquote's raw `bid_spread` / `ask_spread` fields.

**Primary focus symbol** (distributions + day/week/history charts): configurable below (default `XAU/USD`).

**Also covered:** multi-symbol comparison of `bid_spread` / `ask_spread` behaviour, and empirical hypothesis tests for what those fields mean.

Tick rows from `swissquote.bbo` (ALIAS mid/spread/quote_age; same via VIEW `swissquote.bbo_mv`). Uses the same ClickHouse connection as `01_bbo_overview.ipynb`.

**Env:** `CLICKHOUSE_HOST` (and optional user/password/port) via `~/.env` / process env — same as the rest of the project.


In [ ]:
from __future__ import annotations

import json
import urllib.request
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from swissquote.clickhouse import ClickHouseConfig, create_client

from ch_utils import (
    coerce_float_series,
    full_history_bounds,
    latest_trading_day_bounds,
    latest_trading_week_bounds,
    load_bbo_df,
    resample_bbo_chart,
    suggest_resample_rule,
    symbol_availability,
    table_overview,
)

sns.set_theme(style="whitegrid")

cfg = ClickHouseConfig.from_env()
client = create_client(cfg)


def instrument_pip_size(symbol: str) -> float:
    """Conventional pip/point size used to interpret Swissquote spread fields.

    FX XXX/JPY → 0.01; other major FX → 0.0001; XAU → 0.01; XAG → 0.001.
    These are market conventions, not a column Swissquote publishes.
    """
    if symbol.endswith("/JPY") or symbol.startswith("JPY/"):
        return 0.01
    if symbol.startswith("XAU/"):
        return 0.01
    if symbol.startswith("XAG/"):
        return 0.001
    return 0.0001


In [ ]:
overview = table_overview(client)
symbol_counts = pd.DataFrame(overview["symbol_counts"], columns=["symbol", "rows"])
display(symbol_counts.head(15))

# Primary symbol for day/week/history charts (override as needed).
SYMBOL = "XAU/USD"
if SYMBOL not in set(symbol_counts["symbol"]) and not symbol_counts.empty:
    SYMBOL = symbol_counts.iloc[0]["symbol"]

avail = symbol_availability(client, SYMBOL)
print(f"Symbol: {SYMBOL}")
print(f"Rows in ClickHouse:  {avail['row_count']:,}")
print(f"Quote ts range:      {avail['ts_min']} → {avail['ts_max']}")
print(f"Ingestion range:     {avail['ingestion_min']} → {avail['ingestion_max']}")
print("\nLoading full history…")

df = load_bbo_df(
    client,
    symbol=SYMBOL,
    hours=None,
    limit=None,
    enrich=True,
)
print(f"Tick rows loaded:    {len(df):,}")

PROFILE = df["spread_profile"].mode().iloc[0]
DAY_START, DAY_END, DAY_LABEL = latest_trading_day_bounds(df)
WEEK_START, WEEK_END, WEEK_LABEL = latest_trading_week_bounds(df)
ALL_START, ALL_END, ALL_LABEL = full_history_bounds(df)
PIP = instrument_pip_size(SYMBOL)

print(f"\nPlot profile:        {PROFILE}")
print(f"Assumed pip/point:   {PIP}")
print(f"Latest trading day:  {DAY_LABEL}")
print(f"Latest trading week: {WEEK_LABEL}")
print(f"Full history:        {ALL_LABEL}")

df.head(20)


In [ ]:
spread_summary = (
    df.groupby("spread_profile")[["spread_abs", "spread_bps"]]
    .agg(["count", "mean", "median", "std"])
    .round(4)
)
spread_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["spread_bps"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title(f"Spread (bps) — {SYMBOL}")
axes[0].set_xlabel("spread_bps")

sns.boxplot(data=df, x="spread_profile", y="spread_bps", ax=axes[1])
axes[1].set_title("Spread by profile")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
def plot_bbo_window(
    title: str,
    ts_start: pd.Timestamp,
    ts_end: pd.Timestamp,
    label: str,
    *,
    date_formatter: str,
    major_locator,
) -> None:
    rule = suggest_resample_rule(ts_start, ts_end)
    chart_df = resample_bbo_chart(
        df,
        ts_start=ts_start,
        ts_end=ts_end,
        spread_profile=PROFILE,
        rule=rule,
    )
    price_cols = [c for c in ("bid_price", "ask_price", "mid") if c in chart_df.columns]
    spread_note = f"{rule} median (ticks)"
    if chart_df.empty:
        print(f"Skipping {title}: no rows for profile {PROFILE!r}")
        return

    fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
    chart_df[price_cols].plot(ax=axes[0], linewidth=0.9)
    axes[0].set_title(f"{title} — {SYMBOL} ({PROFILE}) | {label}")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left", fontsize=8)

    chart_df["spread_bps"].plot(ax=axes[1], color="crimson", linewidth=0.9)
    axes[1].set_title(f"Spread (bps, {spread_note})")
    axes[1].set_ylabel("bps")
    axes[1].set_xlim(ts_start, ts_end)
    axes[1].xaxis.set_major_locator(major_locator)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter(date_formatter))

    plt.tight_layout()
    plt.show()


plot_bbo_window(
    "One trading day",
    DAY_START,
    DAY_END,
    DAY_LABEL,
    date_formatter="%H:%M",
    major_locator=mdates.HourLocator(interval=2),
)

plot_bbo_window(
    "Trading week (Mon open → Fri close)",
    WEEK_START,
    WEEK_END,
    WEEK_LABEL,
    date_formatter="%a %d",
    major_locator=mdates.DayLocator(),
)

plot_bbo_window(
    "Full history",
    ALL_START,
    ALL_END,
    ALL_LABEL,
    date_formatter="%Y-%m-%d",
    major_locator=mdates.WeekdayLocator(byweekday=mdates.MO, interval=2),
)


In [ ]:
spread = coerce_float_series(df["spread_bps"])
grouped = df.assign(spread_bps=spread).groupby("server")["spread_bps"]
by_server = grouped.agg(rows="count", median="median").join(
    grouped.quantile(0.95).rename("p95")
)
by_server = by_server.sort_values("rows", ascending=False).round(4)
by_server

---

## What are Swissquote `bid_spread` / `ask_spread`?

These columns are **raw vendor fields**, not derived in our pipeline.

| Layer | Name | Notes |
|-------|------|--------|
| Public HTTP JSON | `bidSpread`, `askSpread` | Inside each `spreadProfilePrices[]` entry alongside `bid` / `ask` / `spreadProfile` |
| SBE / codec | `bidSpread`, `askSpread` | `float32` on the wire (`sbe/bbo.xml`); encode reads JSON keys verbatim |
| ClickHouse | `bid_spread`, `ask_spread` | Stored as `Float32`; **not** used by ALIAS `mid` / `spread_abs` / `spread_bps` |

`spread_abs` = `ask_price - bid_price` and `spread_bps` = `spread_abs / mid * 1e4` are **our** derived quote-width metrics. `bid_spread` / `ask_spread` are whatever Swissquote puts on the wire.

Repo references: `cpp/.../codec/encode.cpp` (`profile_obj["bidSpread"]`), fixtures in `cpp/tests/fixtures.hpp` / `python/tests/codec/fixtures.py`, schema `sql/swissquote-bbo.db.sql`. Swissquote does **not** document units in-repo; the next cells test hypotheses empirically.


### Hypotheses

1. **H1 — price half-spreads:** `bid_spread ≈ mid - bid`, `ask_spread ≈ ask - mid` (same units as price).
2. **H2 — absolute width identity:** `bid_spread + ask_spread ≈ ask - bid` (price units).
3. **H3 — pip-scaled width:** `(bid_spread + ask_spread) * pip ≈ ask - bid` for a conventional pip/point size.
4. **H4 — profile markup in pips:** within the same `(platform, server, ts)`, moving from `elite` to a wider profile widens prices by exactly `Δbid_spread * pip` on the bid and `Δask_spread * pip` on the ask.
5. **H5 — stickiness:** values are **stepwise sticky** per `(symbol, platform, server, spread_profile)` (constant for long stretches; occasional level changes) while live `spread_abs` moves continuously.


In [ ]:
# --- Hypothesis checks on the primary symbol (full history already in `df`) ---
work = df.copy()
work["pip"] = PIP
work["half_bid"] = work["mid"] - work["bid_price"]
work["half_ask"] = work["ask_price"] - work["mid"]
work["spread_sum"] = work["bid_spread"] + work["ask_spread"]
work["spread_pips"] = work["spread_abs"] / work["pip"]

h1_bid = (work["bid_spread"] - work["half_bid"]).abs()
h1_ask = (work["ask_spread"] - work["half_ask"]).abs()
h2 = (work["spread_sum"] - work["spread_abs"]).abs()
h3 = (work["spread_sum"] * work["pip"] - work["spread_abs"]).abs()

print(f"Primary symbol={SYMBOL}  pip={PIP}  rows={len(work):,}")
print("\nH1 (price half-spreads): REJECT if medians >> 0")
print(f"  median |bid_spread - (mid-bid)| = {h1_bid.median():.6g}")
print(f"  median |ask_spread - (ask-mid)| = {h1_ask.median():.6g}")

print("\nH2 (sum == ask-bid in price units): REJECT if median >> 0")
print(f"  median |bid_spread+ask_spread - spread_abs| = {h2.median():.6g}")

print("\nH3 (sum * pip == ask-bid):")
print(f"  median |(bid_spread+ask_spread)*pip - spread_abs| = {h3.median():.6g}")
print(f"  median rel err vs spread_abs = {(h3 / work['spread_abs'].clip(lower=1e-12)).median():.2%}")

# H5 stickiness
nunique = (
    work.groupby(["platform", "server", "spread_profile"], observed=True)["bid_spread"]
    .nunique()
    .rename("nunique_bid_spread")
)
sym_rate = float((work["bid_spread"] == work["ask_spread"]).mean())
print("\nH5 (stickiness / symmetry):")
print(f"  median nunique(bid_spread) per venue×profile = {nunique.median():.1f}")
print(f"  max nunique(bid_spread) = {int(nunique.max())}")
print(f"  fraction bid_spread == ask_spread = {sym_rate:.2%}")

field_vs_live = (
    work.groupby("spread_profile", observed=True)
    .agg(
        rows=("spread_abs", "size"),
        med_bid_spread=("bid_spread", "median"),
        med_ask_spread=("ask_spread", "median"),
        med_field_sum_pips=("spread_sum", "median"),
        med_live_spread_pips=("spread_pips", "median"),
        med_spread_abs=("spread_abs", "median"),
    )
    .round(4)
)
field_vs_live


In [ ]:
# H4: profile markup vs elite, matched on (platform, server, ts)
elite = work.loc[work["spread_profile"] == "elite", [
    "ts", "platform", "server", "bid_price", "ask_price", "bid_spread", "ask_spread"
]].rename(
    columns={
        "bid_price": "elite_bid",
        "ask_price": "elite_ask",
        "bid_spread": "elite_bs",
        "ask_spread": "elite_as",
    }
)

merged = work.merge(elite, on=["ts", "platform", "server"], how="inner")
merged["d_bs"] = merged["bid_spread"] - merged["elite_bs"]
merged["d_as"] = merged["ask_spread"] - merged["elite_as"]
merged["d_bid_pips"] = (merged["elite_bid"] - merged["bid_price"]) / merged["pip"]
merged["d_ask_pips"] = (merged["ask_price"] - merged["elite_ask"]) / merged["pip"]
merged["err_bid"] = (merged["d_bid_pips"] - merged["d_bs"]).abs()
merged["err_ask"] = (merged["d_ask_pips"] - merged["d_as"]).abs()

print(f"Matched rows vs elite: {len(merged):,}")
print("H4 median |Δprice/pip - Δbid_spread| (should be ~0 if fields are pip markups):")
print(merged[["err_bid", "err_ask"]].median().to_string())
print("H4 p95:")
print(merged[["err_bid", "err_ask"]].quantile(0.95).to_string())

by_profile = (
    merged.groupby("spread_profile", observed=True)
    .agg(
        rows=("err_bid", "size"),
        med_err_bid=("err_bid", "median"),
        med_err_ask=("err_ask", "median"),
        med_d_bs=("d_bs", "median"),
        med_d_bid_pips=("d_bid_pips", "median"),
    )
    .round(6)
)
by_profile


In [ ]:
# Visual: field sum (pips) vs live spread in pips; stickiness over a trading day
day = work.loc[(work["ts"] >= DAY_START) & (work["ts"] <= DAY_END)].copy()
day_prof = day.loc[day["spread_profile"] == PROFILE]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.scatterplot(
    data=work.sample(min(8_000, len(work)), random_state=0),
    x="spread_sum",
    y="spread_pips",
    hue="spread_profile",
    s=12,
    ax=axes[0],
    alpha=0.5,
)
lo = float(min(work["spread_sum"].min(), work["spread_pips"].min()))
hi = float(max(work["spread_sum"].max(), work["spread_pips"].max()))
axes[0].plot([lo, hi], [lo, hi], ls="--", c="gray", lw=1, label="y=x")
axes[0].set_title(f"{SYMBOL}: live pips vs field sum")
axes[0].set_xlabel("bid_spread + ask_spread")
axes[0].set_ylabel("spread_abs / pip")
axes[0].legend(fontsize=7, loc="best")

sns.boxplot(data=work, x="spread_profile", y="bid_spread", ax=axes[1])
axes[1].set_title("bid_spread by profile (sticky levels)")
axes[1].tick_params(axis="x", rotation=30)

if not day_prof.empty:
    sample = day_prof.sort_values("ts").iloc[:: max(1, len(day_prof) // 800)]
    axes[2].plot(sample["ts"], sample["spread_pips"], lw=0.8, label="live spread (pips)")
    axes[2].plot(sample["ts"], sample["spread_sum"], lw=1.2, label="bid+ask spread fields")
    axes[2].set_title(f"Day stickiness — {PROFILE}")
    axes[2].legend(fontsize=8)
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
else:
    axes[2].set_title("No day rows for profile")

plt.tight_layout()
plt.show()

print(
    "If H4 holds and H1/H2 fail: fields are pip/point side spreads (markups), "
    "not price distances. Live width still needs spread_abs / spread_bps."
)


### Interim reading (primary symbol)

- **Reject H1 / H2:** `bid_spread` / `ask_spread` are **not** price-unit half-spreads and do **not** sum to `ask - bid`.
- **H3 is only approximate:** `(bid_spread + ask_spread) * pip` is usually *near* live width for FX after adding the elite market width, but it is **not** an identity on every tick (especially metals).
- **H4 is the strong positive result:** profile upgrades/downgrades move bid/ask by `Δspread * pip` versus `elite` on the same venue timestamp (≈ machine precision on major FX in short windows; small residuals on JPY/metals and over long histories when field levels step).
- **H5:** values are **symmetric** (`bid_spread == ask_spread` in this dataset) and **stepwise sticky** per venue×profile while `spread_abs` fluctuates every tick.

So the fields behave as **Swissquote's per-side spread figures in instrument pips/points** (account-profile markups over the core/`elite` book for FX; metals also carry a non-zero elite base in the same units). Use `spread_abs` / `spread_bps` for realized quote width.


---

## Multi-symbol comparison

Load a short recent window for several liquid symbols and compare:
- level / stickiness of `bid_spread` by profile
- H1–H4 summary metrics
- how `bid_spread + ask_spread` relates to live spread in pips


In [ ]:
COMPARE_SYMBOLS = [
    "EUR/USD",
    "GBP/USD",
    "AUD/USD",
    "USD/CHF",
    "USD/JPY",
    "EUR/JPY",
    "XAU/USD",
    "XAG/USD",
]
COMPARE_HOURS = 6  # keep memory modest; full-history primary symbol already loaded above

available = set(symbol_counts["symbol"])
missing = [s for s in COMPARE_SYMBOLS if s not in available]
if missing:
    print("Not in ClickHouse (skipped):", ", ".join(missing))

frames: list[pd.DataFrame] = []
for sym in COMPARE_SYMBOLS:
    if sym not in available:
        continue
    part = load_bbo_df(client, symbol=sym, hours=COMPARE_HOURS, limit=None, enrich=True)
    print(f"{sym}: {len(part):,} rows")
    frames.append(part)

multi = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nMulti-symbol rows: {len(multi):,}  symbols: {multi['symbol'].nunique() if len(multi) else 0}")

if multi.empty:
    raise RuntimeError("No multi-symbol rows loaded — check COMPARE_SYMBOLS / ClickHouse.")

multi = multi.copy()
multi["pip"] = multi["symbol"].map(instrument_pip_size)
multi["half_bid"] = multi["mid"] - multi["bid_price"]
multi["spread_sum"] = multi["bid_spread"] + multi["ask_spread"]
multi["spread_pips"] = multi["spread_abs"] / multi["pip"]
multi.head()


In [ ]:
def h4_errors(g: pd.DataFrame) -> pd.Series:
    """Median markup identity errors vs elite within this symbol slice."""
    elite = g.loc[g["spread_profile"] == "elite", [
        "ts", "platform", "server", "bid_price", "ask_price", "bid_spread", "ask_spread", "pip"
    ]]
    if elite.empty:
        return pd.Series({"h4_med_err_bid": np.nan, "h4_med_err_ask": np.nan, "h4_n": 0})
    e = elite.rename(columns={
        "bid_price": "elite_bid", "ask_price": "elite_ask",
        "bid_spread": "elite_bs", "ask_spread": "elite_as",
    }).drop(columns=["pip"], errors="ignore")
    m = g.merge(e, on=["ts", "platform", "server"], how="inner")
    pip = m["pip"]
    err_bid = ((m["elite_bid"] - m["bid_price"]) / pip - (m["bid_spread"] - m["elite_bs"])).abs()
    err_ask = ((m["ask_price"] - m["elite_ask"]) / pip - (m["ask_spread"] - m["elite_as"])).abs()
    return pd.Series({
        "h4_med_err_bid": err_bid.median(),
        "h4_med_err_ask": err_ask.median(),
        "h4_n": len(m),
    })


hyp = []
for sym, g in multi.groupby("symbol", observed=True):
    h4 = h4_errors(g)
    hyp.append({
        "symbol": sym,
        "pip": instrument_pip_size(sym),
        "rows": len(g),
        "h1_med_|bs-(mid-bid)|": (g["bid_spread"] - g["half_bid"]).abs().median(),
        "h2_med_|sum-spread_abs|": (g["spread_sum"] - g["spread_abs"]).abs().median(),
        "h3_med_rel_|(sum*pip)-abs|/abs": (
            (g["spread_sum"] * g["pip"] - g["spread_abs"]).abs() / g["spread_abs"].clip(lower=1e-12)
        ).median(),
        "sym_rate": (g["bid_spread"] == g["ask_spread"]).mean(),
        "med_nunique_bs": g.groupby(
            ["platform", "server", "spread_profile"], observed=True
        )["bid_spread"].nunique().median(),
        **h4,
    })

hyp_df = pd.DataFrame(hyp).set_index("symbol").sort_index()
print("Cross-symbol hypothesis summary (last", COMPARE_HOURS, "hours):")
hyp_df.round(6)


In [ ]:
profile_levels = (
    multi.groupby(["symbol", "spread_profile"], observed=True)
    .agg(
        med_bid_spread=("bid_spread", "median"),
        med_field_sum=("spread_sum", "median"),
        med_live_pips=("spread_pips", "median"),
        med_spread_bps=("spread_bps", "median"),
    )
    .reset_index()
)

# Live pips ≈ field sum + elite live residual (FX); show both
elite_live = (
    profile_levels.loc[profile_levels["spread_profile"] == "elite", ["symbol", "med_live_pips"]]
    .rename(columns={"med_live_pips": "elite_live_pips"})
)
profile_levels = profile_levels.merge(elite_live, on="symbol", how="left")
profile_levels["field_sum_plus_elite_live"] = (
    profile_levels["med_field_sum"] + profile_levels["elite_live_pips"]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=profile_levels,
    x="symbol",
    y="med_bid_spread",
    hue="spread_profile",
    ax=axes[0],
)
axes[0].set_title("Median bid_spread by symbol × profile")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylabel("bid_spread (pip/point units)")

plot_df = profile_levels.melt(
    id_vars=["symbol", "spread_profile"],
    value_vars=["med_live_pips", "med_field_sum", "field_sum_plus_elite_live"],
    var_name="metric",
    value_name="pips",
)
# Keep one profile for readability
focus_profile = "premium" if (profile_levels["spread_profile"] == "premium").any() else PROFILE
sns.barplot(
    data=plot_df.loc[plot_df["spread_profile"] == focus_profile],
    x="symbol",
    y="pips",
    hue="metric",
    ax=axes[1],
)
axes[1].set_title(f"Live vs field pips — {focus_profile}")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

profile_levels.sort_values(["symbol", "spread_profile"]).round(4)


In [ ]:
# Optional: live public API snapshot (no ClickHouse) — useful if DB is down / for spot-check
API_SYMBOLS = ["EUR/USD", "USD/JPY", "XAU/USD", "XAG/USD"]
api_rows = []
for sym in API_SYMBOLS:
    url = f"https://forex-data-feed.swissquote.com/public-quotes/bboquotes/instrument/{sym}"
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            payload = json.load(resp)
    except Exception as exc:  # noqa: BLE001 — notebook resilience
        print(f"{sym}: API fetch failed ({exc})")
        continue
    pip = instrument_pip_size(sym)
    for venue in payload:
        topo = venue.get("topo", {})
        for prof in venue.get("spreadProfilePrices", []):
            bid = float(prof["bid"])
            ask = float(prof["ask"])
            bs = float(prof["bidSpread"])
            a_s = float(prof["askSpread"])
            api_rows.append({
                "symbol": sym,
                "platform": topo.get("platform"),
                "server": topo.get("server"),
                "spread_profile": prof["spreadProfile"],
                "bid": bid,
                "ask": ask,
                "spread_abs": ask - bid,
                "bidSpread": bs,
                "askSpread": a_s,
                "sum": bs + a_s,
                "live_pips": (ask - bid) / pip,
            })

api_df = pd.DataFrame(api_rows)
if api_df.empty:
    print("No API rows — skip live cross-check.")
else:
    print("Live API cross-check (raw JSON bidSpread/askSpread):")
    display(
        api_df.groupby(["symbol", "spread_profile"], observed=True)[
            ["bidSpread", "askSpread", "sum", "live_pips", "spread_abs"]
        ]
        .median()
        .round(4)
    )


---

## Conclusion: best interpretation of `bid_spread` / `ask_spread`

**Best interpretation**

Swissquote's raw `bidSpread` / `askSpread` (stored as `bid_spread` / `ask_spread`) are **per-side spread figures in instrument pips/points**, published per `spreadProfile`, **not** distances in price units.

Evidence:
1. **Codec/docs:** pass-through from public JSON → SBE float32 → ClickHouse; no unit conversion in our code.
2. **Reject price identities (H1/H2):** magnitudes match “pips” (e.g. `0.65` on EUR/USD premium, `25.4` on XAU/USD), not `mid - bid`.
3. **Accept markup identity (H4):** on a fixed venue timestamp,
   \(\Delta bid\_price / pip = \Delta bid\_spread\) (and likewise on the ask) versus `elite`, to float precision on major FX.
4. **Stickiness (H5):** stepwise sticky per symbol×venue×profile while live `spread_abs` moves every tick; bid/ask fields are symmetric in this dataset.
5. **FX pattern:** `elite` fields are `0`; wider profiles add the familiar advertised half-spreads (≈0.4–0.85 per side). Live width in pips ≈ elite market width + `bid_spread + ask_spread`.
6. **Metals:** same pip-space markup math between profiles, but `elite` fields are **non-zero** and **do not** equal live half-spread in points — treat the absolute level as Swissquote's published point figure, and still use `spread_abs` for realized width.

**Implications for spread analysis**
- Prefer **`spread_abs` / `spread_bps`** for quote quality, time series, and venue comparisons.
- Use **`bid_spread` / `ask_spread`** when studying **account-profile markup** in pip space, or when mirroring Swissquote's own profile labelling.
- Do not add `bid_spread + ask_spread` to prices without multiplying by the instrument pip/point size.

**Residual uncertainty**
- Swissquote does not publish an official unit glossary for these JSON keys in this repo or the public feed docs we could find.
- Exact pip/point convention for every CFD/metal is assumed from market convention (`XAU` 0.01, `XAG` 0.001, JPY 0.01, other FX 0.0001); exotic symbols may differ.
- Absolute metal field levels are sticky and only loosely track live half-spread — the economic meaning of that elite base (floor, smoothed typical, display number) is not independently documented.
- Occasional sub-pip residuals on JPY/metals likely from price quantization / float32 storage of the spread fields.
